# 02: Per-layer CKA, by SNR

**What this shows.** Two views of where MUSE adapts in its layer stack: a layer x SNR heatmap of mean CKA, and a per-layer SNR-sensitivity profile (linear regression slope of CKA vs SNR).

**What to look for.** Encoder-1/2 stay near 1.0 across all SNRs (low slope, low sensitivity). Latent and decoder layers drop with worsening SNR (negative slope, high sensitivity). The refinement block partly recovers — that's the qualitative signature reproduced in the paper.

**Runtime.** <30 s on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, MODEL_LABELS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## Load MUSE CKA-vs-SNR table

The norm1 layers (one per transformer layer) are the ones used in the paper. We pull them from the demo parquet itself so the notebook stays self-contained.

In [ ]:
muse_path = DATA_DIR / ("snr/cka_snr_muse.parquet" if (DATA_DIR / "snr").exists() else "cka_snr_muse_demo.parquet")
cka_df = pd.read_parquet(muse_path)

norm1_layers = sorted(L for L in cka_df["layer"].unique() if L.endswith(".norm1"))
block_order = ["encoder_level1", "encoder_level2", "latent", "decoder_level2", "decoder_level1", "mag_refinement"]
norm1_layers.sort(key=lambda L: (next((i for i, b in enumerate(block_order) if b in L), len(block_order)), L))
print(f"norm1 layers: {len(norm1_layers)}")

cka_n1 = cka_df[cka_df["layer"].isin(norm1_layers)].copy()
mean_cka_df = cka_n1.groupby(["layer", "snr"], as_index=False)["CKA"].mean()
mean_cka_df["layer"] = pd.Categorical(mean_cka_df["layer"], categories=norm1_layers, ordered=True)

## Heatmap: layer x SNR

Rows are layers (encoder → latent → decoder → refinement), columns are SNR. Yellow = clean-like representations; dark = strongly perturbed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

heat = mean_cka_df.pivot(index="layer", columns="snr", values="CKA").loc[norm1_layers]
fig, ax = plt.subplots(figsize=(8.5, 3.8))
sns.heatmap(heat.T, cmap="viridis", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "Mean CKA", "shrink": 0.9})
ax.invert_yaxis()
ax.set_xlabel("Block")
ax.set_ylabel("SNR (dB)")
ax.set_title("Mean CKA by layer and SNR (MUSE, norm1 layers)")

block_size = max(1, len(norm1_layers) // 6)
tick_labels = ["Enc-L1", "Enc-L2", "Latent", "Dec-L2", "Dec-L1", "Refinement"]
tick_positions = [i * block_size + block_size / 2 for i in range(6)]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=30, ha="center")
for i in range(block_size, len(norm1_layers), block_size):
    ax.axvline(x=i, color="k", linestyle="--", linewidth=0.8)
plt.tight_layout();

## Per-layer SNR sensitivity (regression slope)

For each layer we regress mean CKA on SNR. Layers with slope near 0 are robust to SNR; large positive slopes mark layers where representations track the input degradation.

In [ ]:
from sklearn.linear_model import LinearRegression

rows = []
for layer in norm1_layers:
    g = mean_cka_df[mean_cka_df["layer"] == layer].sort_values("snr")
    x, y = g["snr"].to_numpy(dtype=float), g["CKA"].to_numpy(dtype=float)
    if len(x) < 2:
        continue
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    rows.append({"layer": layer, "slope": reg.coef_[0], "intercept": reg.intercept_,
                  "r2": reg.score(x.reshape(-1, 1), y)})
sens = pd.DataFrame(rows)
print(f"layers fit: {len(sens)} | mean R²: {sens['r2'].mean():.3f}")
sens.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(sens))
ax.plot(x, sens["slope"].values, marker="o", color=MODEL_COLORS["muse"], label="slope")
ax.axhline(0, color="k", linestyle=":", linewidth=0.8)
ax.set_xlabel("Block")
ax.set_ylabel("d CKA / d SNR")
ax.set_title(f"SNR sensitivity per norm1 layer ({MODEL_LABELS['muse']})")
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=30, ha="center")
for i in range(block_size, len(sens), block_size):
    ax.axvline(x=i - 0.5, color="k", linestyle="--", linewidth=0.6)
plt.tight_layout();